# SmartHire — Supervised Shortlisting / Fit Predictor (Model B)

Predicts candidate shortlisting probability based on engineered features: skill overlap, experience match, education match, and text similarity. Compares Logistic Regression vs Random Forest vs XGBoost.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import CLEAN_RESUMES_PATH, CLEAN_JOBS_PATH
from src.models.fit_predictor import train_fit_predictor, predict_fit_score
from src.features.text_features import load_tfidf_vectorizer

resumes = pd.read_csv(CLEAN_RESUMES_PATH)
jobs = pd.read_csv(CLEAN_JOBS_PATH)
vectorizer = load_tfidf_vectorizer()


### 1. Train Models & Compare Metrics


In [ ]:
pipe, results = train_fit_predictor(resumes_df=resumes, jobs_df=jobs, vectorizer=vectorizer, save=True)
comp_table = pd.DataFrame(results['comparison_table']).T
comp_table[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]


### 2. Feature Importances


In [ ]:
feat_imp = pd.Series(results['feature_importances']).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
feat_imp.plot(kind='barh', color='teal')
plt.title("Fit Predictor Feature Importances")
plt.xlabel("Weight / Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### 3. Inference on Sample Candidate-Job Pair


In [ ]:
cand_cv = "Data Scientist with expertise in Python, Machine Learning, Scikit-learn, Pandas, SQL and Tableau with 3 years experience."
job_req = "Senior Data Analyst required. Must have 3+ years experience in Python, SQL, Tableau, data visualization and statistical modeling."

fit_out = predict_fit_score(cand_cv, job_req, model=pipe, vectorizer=vectorizer)
print("Fit Probability:", fit_out['fit_percent'], "%")
print("Shortlist Status:", fit_out['status'])
print("Feature Breakdown:", fit_out['features'])
